# Amazon Customer Behavior Analysis

## Task 1: Data Cleaning and Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [2]:
df = pd.read_csv("../data/Amazon.csv")

print("Dataset Loaded Successfully!")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset Loaded Successfully!
Rows: 800
Columns: 24


In [3]:
# Initial Inspection of dataset

df.head()
df.info()

# Summary statistics
df.describe(include="all").T

# Check column names
print(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 24 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   Timestamp                               800 non-null    object
 1   age                                     800 non-null    int64 
 2   Gender                                  800 non-null    object
 3   Purchase_Frequency                      800 non-null    object
 4   Purchase_Categories                     800 non-null    object
 5   Personalized_Recommendation_Frequency   800 non-null    object
 6   Browsing_Frequency                      800 non-null    object
 7   Product_Search_Method                   643 non-null    object
 8   Search_Result_Exploration               800 non-null    object
 9   Customer_Reviews_Importance             800 non-null    int64 
 10  Add_to_Cart_Browsing                    800 non-null    object
 11  Cart_C

In [4]:
# Renaming Incorrect columns 
df.rename(columns={
    "Personalized_Recommendation_Frequency ": "Personalized_Recommendation_Score",
    "Rating_Accuracy ": "Rating_Accuracy"
}, inplace=True)

df.columns = df.columns.str.strip()

print(df.columns.tolist())

['Timestamp', 'age', 'Gender', 'Purchase_Frequency', 'Purchase_Categories', 'Personalized_Recommendation_Frequency', 'Browsing_Frequency', 'Product_Search_Method', 'Search_Result_Exploration', 'Customer_Reviews_Importance', 'Add_to_Cart_Browsing', 'Cart_Completion_Frequency', 'Cart_Abandonment_Factors', 'Saveforlater_Frequency', 'Review_Left', 'Review_Reliability', 'Review_Helpfulness', 'Personalized_Recommendation_Score', 'Recommendation_Helpfulness', 'Rating_Accuracy', 'Shopping_Satisfaction', 'Service_Appreciation', 'Improvement_Areas', 'transaction']


In [5]:
# Count the number of completely duplicate rows
duplicate_rows = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [6]:
# Display rows with duplicate transaction IDs
duplicate_transactions = df[df.duplicated(subset="transaction", keep=False)]

duplicate_transactions[["transaction", "Timestamp"]]

,transaction,Timestamp
121,720840,2023/06/06 4:46:25 PM GMT+5:30
408,295647,2023/06/06 6:45:33 PM GMT+5:30
489,295647,2023/06/06 6:35:14 PM GMT+5:30
561,720840,2023/06/11 11:10:14 PM GMT+5:30


In [7]:
# Compare the duplicate transaction records
duplicate_transactions

,Timestamp,age,Gender,Purchase_Frequency,Purchase_Categories,Personalized_Recommendation_Frequency,Browsing_Frequency,Product_Search_Method,Search_Result_Exploration,Customer_Reviews_Importance,...,Review_Left,Review_Reliability,Review_Helpfulness,Personalized_Recommendation_Score,Recommendation_Helpfulness,Rating_Accuracy,Shopping_Satisfaction,Service_Appreciation,Improvement_Areas,transaction
121,2023/06/06 4:46:25 PM GMT+5:30,67,Male,Few times a month,Beauty and Personal Care;Clothing and Fashion;...,Sometimes,Few times a week,Keyword,First page,5,...,No,Heavily,No,2,Sometimes,5,1,Customer service,Quality of product is very poor according to t...,720840
408,2023/06/06 6:45:33 PM GMT+5:30,26,Female,Less than once a month,Beauty and Personal Care,Sometimes,Few times a week,Keyword,First page,5,...,Yes,Occasionally,No,5,No,5,2,Customer service,Customer service responsiveness,295647
489,2023/06/06 6:35:14 PM GMT+5:30,5,Female,Few times a month,Groceries and Gourmet Food;Home and Kitchen;ot...,Sometimes,Rarely,NaN,Multiple pages,4,...,No,Rarely,No,3,Sometimes,4,4,User-friendly website/app interface,.,295647
561,2023/06/11 11:10:14 PM GMT+5:30,55,Others,Once a month,Home and Kitchen;others,Yes,Rarely,NaN,First page,3,...,Yes,Occasionally,Yes,1,Yes,3,2,Customer service,Customer service responsiveness,720840


**Observation:**
- Two transaction IDs appear more than once in the dataset.
- However, the corresponding survey responses are different.
- This suggests transaction ID collisions rather than duplicate records.
- Therefore, these records are retained for further analysis.

In [8]:
# Count missing values in each column
missing_values = df.isnull().sum()

# Display only columns with missing values
missing_values[missing_values > 0]

Product_Search_Method    157
dtype: int64

In [9]:
# Filling missing values with 'Not Specified'

df["Product_Search_Method"] = df["Product_Search_Method"].fillna("Not Specified")

# Verifying that no missing values remained
print(df["Product_Search_Method"].isnull().sum())

0


**Observation:**
- The 'Product_Search_Method' column contained 157 missing values.
- These were replaced with 'Not Specified' to preserve all survey responses.

In [10]:
# Standardizing categorical Values

categorical_columns = [
    "Gender",
    "Purchase_Frequency",
    "Personalized_Recommendation_Frequency",
    "Browsing_Frequency",
    "Product_Search_Method",
    "Search_Result_Exploration",
    "Add_to_Cart_Browsing",
    "Cart_Completion_Frequency",
    "Saveforlater_Frequency",
    "Review_Left",
    "Review_Reliability",
    "Review_Helpfulness",
    "Recommendation_Helpfulness",
    "Service_Appreciation"
]

# Removing the leading/trailing spaces and standardize text formatting
for column in categorical_columns:
    df[column] = df[column].astype(str).str.strip().str.title()

In [11]:
# Cleaning inconsistent values

df["Service_Appreciation"] = df["Service_Appreciation"].replace(".", "Not Specified")

df["Improvement_Areas"] = (
    df["Improvement_Areas"]
    .str.strip()
    .replace(".", "Not Specified")
)

In [12]:
# Converting to numeric columns

numeric_columns = [
    "age",
    "Customer_Reviews_Importance",
    "Personalized_Recommendation_Score",
    "Rating_Accuracy",
    "Shopping_Satisfaction"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [13]:
# Convert Timestamp to datetime

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"].str.replace(" GMT", "", regex=False),
    errors="coerce"
)

C:\Users\anant\AppData\Local\Temp\ipykernel_14460\728541249.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Timestamp"] = pd.to_datetime(


In [14]:
df["Timestamp"].head()

0   2023-06-08 19:50:55+05:30
1   2023-06-09 09:37:44+05:30
2   2023-06-11 23:26:54+05:30
3   2023-06-08 17:17:10+05:30
4   2023-06-11 22:59:30+05:30
Name: Timestamp, dtype: datetime64[ns, UTC+05:30]

In [15]:
# Creating a list from purchase categories

df["Purchase_Categories_List"] = df["Purchase_Categories"].str.split(";")

In [16]:
# Verifying the cleaned dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype                    
---  ------                                 --------------  -----                    
 0   Timestamp                              800 non-null    datetime64[ns, UTC+05:30]
 1   age                                    800 non-null    int64                    
 2   Gender                                 800 non-null    object                   
 3   Purchase_Frequency                     800 non-null    object                   
 4   Purchase_Categories                    800 non-null    object                   
 5   Personalized_Recommendation_Frequency  800 non-null    object                   
 6   Browsing_Frequency                     800 non-null    object                   
 7   Product_Search_Method                  800 non-null    object                   
 8   Search_Result_Exploration     

In [17]:
# Saving cleaned dataset

df.to_csv("../data/Amazon_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
